# Agent

In [ ]:
from dataclasses import dataclass
import requests
from ddgs import DDGS
import lisette

from llm_sandbox_ui.app_config import (
    KERNEL_URL, MODEL, NOTEBOOK_SYS_PROMPT, SYS_PROMPT,
    CODE_GENERATOR, CODE_IMPROVER, UNIT_TEST_STR, SYS_CODER, SYS_REVIEW
)
from llm_sandbox_ui.kernel import *


ModuleNotFoundError: No module named 'llm_sandbox_ui'

In [ ]:

@dataclass
class SentimentChecker:
    """ Use this tool register whether you think the code review your looking at approved\n
    """
    approved: bool
    

In [ ]:

class AgentTools():
    def __init__(self, stream, chat, prompt, cell_id, code = '', print_updates = False):
        """Initialize agent tools.
        
        Args:
            stream: SSEStream for output to GUI.
            chat: LLM chat instance.
            prompt: User's code generation request.
            cell_id: Unique cell identifier.
            code: Existing code to improve.
            print_updates: Enable debug printing.
        """
        self.chat = chat
        self.code = code
        self.prompt = prompt
        self.cell_id = cell_id
        self.print_updates = print_updates
        self.stream = stream

    def print_update(self, text):
        """Print debug message if updates enabled."""
        if self.print_updates:
            print(text)

    def collect(self, piece, dtype='text'):
        """Send output to GUI stream.
        
        Args:
            piece: Content to send.
            dtype: One of 'text', 'tag', or 'output'.
        """
        if dtype == 'text':
            self.stream.text(piece)
        elif dtype == 'tag':
            self.stream.tag(piece)
        elif dtype == 'output':
            self.stream.output(piece)

    def chat_history_swap(self,new_sys,step_prompt):
        """Get LLM response with temporary system prompt.
        
        Args:
            new_sys: Temporary system prompt.
            step_prompt: Prompt for this step.
        
        Returns:
            Collected response text.
        """
        old_sys = self.chat.sp
        self.chat.sp = new_sys
        old_hist = self.chat.hist[:]
        
        response = self.chat(step_prompt,stream=True)
        collected_stream = self.collect_llm_stream(response)
        
        self.chat.sp = old_sys
        self.chat.hist = old_hist
        return collected_stream

    def collect_llm_stream(self,response):
        """Stream LLM chunks to GUI.
        
        Args:
            response: LLM streaming response.
        
        Returns:
            Full accumulated text.
        """
        raw = ''
        for piece in response:
            if self.stream.aborted():  
                break
            if hasattr(piece, 'choices'):
                if hasattr(piece.choices[0], 'delta'):
                    if hasattr(piece.choices[0].delta, 'content'):
                        crumb = piece.choices[0].delta.content
                        if crumb:
                            raw += crumb
                            self.collect(crumb)
        return raw

    def search_web(self, query: str, max_results: int = 5):
        """Search DuckDuckGo for query.
        
        Args:
            query: Search string.
            max_results: Max results to return.
        """
        from ddgs import DDGS
        self.print_update('Searching Web\n')
        self.collect('tool-websearch',dtype='tag')

        results = DDGS().text(query, max_results=max_results)
    
        snippet = str([{"title": r["title"], "url": r["href"], "snippet": r["body"]} 
                for r in results])
        result = str({'step':'SearchWeb','code':snippet})
        self.chat.hist.append({"role":"assistant","content":result})
        return 


    def read_url(self, url:str):
        """Fetch webpage content.
        
        Args:
            url: URL to retrieve.
        
        Returns:
            HTML content.
        """
        import requests
        self.print_update('Loading URL\n')
        self.collect('tool-readurl',dtype='tag')

        response = requests.get(url)
        html = response.text

        result = str({'step':'ReadUrl','code':html})
        self.chat.hist.append({"role":"assistant","content":result})
        return html


    def improve_code(self):
        """Improve existing code based on unit test feedback. Returns improved code."""
        self.print_update('Improving Code\n')
        self.collect('code-box',dtype='tag')

        raw_code = self.chat_history_swap(SYS_CODER,CODE_IMPROVER)
        self.code = raw_code

        result = str({'step':'CodeImprovment','code':self.code})
        self.chat.hist.append({"role":"user","content":result})

        return result


    def generate_code(self):
        """Generate initial code and unit test for the given prompt. Returns generated code."""
        self.print_update('Generating Code\n')
        self.collect('code-box>',dtype='tag')

        raw_code = self.chat_history_swap(SYS_CODER,CODE_GENERATOR)
        self.code = raw_code

        result = str({'step':'CodeFirstPass','code':self.code})
        self.chat.hist.append({"role":"user","content":result})
        return result


    def unit_test(self):
        """Execute code in Unreal Engine and review results. Returns test outcome with pass/fail status."""
        self.print_update('Testing Code\n')
        self.collect('unit-box',dtype='tag')

        unit_test_result, unit_test_result_ansi = execute_unreal_code(self.code)
        
        # Send to GUi
        accumulated = convert_to_accumulated(unit_test_result_ansi)
        self.collect(accumulated, dtype='output')

        unit_test_dict = {'unit_test_result':unit_test_result,
                          'step':'CodeUnitTest'}
                          
        self.chat.hist.append({"role":"assistant",
                               "content":str(unit_test_dict)})

        self.collect('review-box',dtype='tag')
        raw_review = self.chat_history_swap(SYS_REVIEW,UNIT_TEST_STR)

        boolean_query =[{"role":"user", 
                        "content":f'Is this response saying this code is good enough to approve?: {raw_review}'}]
        sent_result = lisette.structured(MODEL,
                                boolean_query,
                                tool=SentimentChecker)

        self.print_update(str(sent_result.approved)+'\n')
        if sent_result.approved:
            self.collect("DONE",dtype='tag')

        result = str({'pass':sent_result.approved,
                    'explanation':raw_review,
                    'step':'CodeUnitTestReview'}) 

        self.chat.hist.append({"role":"assistant","content":result})
        return result

    def get_tools(self):
        return [self.improve_code,self.generate_code,self.unit_test,self.read_url,self.search_web]
        